In [31]:
import os
os.environ['KMP_DUPLICATE_LIB_OK']='True'
os.environ["TF_XLA_FLAGS"] = "--tf_xla_auto_jit=2 --tf_xla_cpu_global_jit"
os.environ[
    "XLA_FLAGS"
] = "--xla_gpu_cuda_data_dir=/hpc/mp/spack/opt/spack/linux-ubuntu20.04-zen2/gcc-10.3.0/cuda-11.4.4-ctldo35wmmwws3jbgwkgjjcjawddu3qz/"


import numpy as np
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Conv1D, MaxPooling1D, GlobalAveragePooling1D, GlobalAveragePooling2D
from tensorflow.keras.layers import Flatten, Dense, Dropout
from tensorflow.keras.layers import Embedding, Input, Concatenate
from tensorflow.keras.layers import Subtract
from tensorflow.keras.utils import plot_model
import tensorflow as tf

from keras import backend as K

from tensorflow.keras.layers import MultiHeadAttention, LayerNormalization
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Layer

from tensorflow.keras.layers import Multiply, Add, Lambda

from tensorflow.keras.optimizers.legacy import Adam
from tensorflow.keras.optimizers.schedules import ExponentialDecay

from tensorflow.keras.layers import (
    Layer,
    LeakyReLU,
    PReLU,
    Add,
    UpSampling2D,
    Activation,
    SpatialDropout2D,
    Conv2D,
    SeparableConv2D,
    BatchNormalization,
    Concatenate,
    Flatten,
    Dense,
)

from tensorflow import pad

from scripts.config import SimConfig
from scripts.dataloader import DataLoader

from sklearn import metrics as mt
from matplotlib import pyplot as plt
%matplotlib inline

In [32]:
s = SimConfig("settings/ul_nn_256.json")

batch_size = 8
max_epochs = 500

# load data as a generator so we do not need to have it all in memory
d = tf.data.Dataset.from_generator(
    DataLoader(s.data_file_complete),
    output_signature=(
        tf.TensorSpec(shape=(s.nside, s.nside, 1), dtype=s.r_dtype),
        tf.TensorSpec(shape=(), dtype=s.r_dtype),
    ),
)

# fix a log warning
options = tf.data.Options()
options.experimental_distribute.auto_shard_policy = (
    tf.data.experimental.AutoShardPolicy.DATA
)
d = d.with_options(options)

Loaded settings from file: settings/ul_nn_256.json
{ 'alm_cache_dir': 'data/alm_cache',
  'base_dir': 'data',
  'beam_width': 1.4,
  'cosmo_params': { 'AccuracyBoost': 2.0,
                    'As': 2.13e-09,
                    'DoLateRadTruncation': False,
                    'H0': 67.5,
                    'TCMB': 2.7255,
                    'lAccuracyBoost': 2.0,
                    'lSampleBoost': 2.0,
                    'lens_potential_accuracy': 2,
                    'lmax': 750,
                    'max_l': 1000,
                    'mnu': 0.06,
                    'ns': 0.9624,
                    'ombh2': 0.02233,
                    'omch2': 0.1198,
                    'pivot_scalar': 0.05,
                    'r': 0,
                    'tau': 0.0561},
  'debug': True,
  'disable_noise': True,
  'double_precision': False,
  'duplicate_backgrounds': 10,
  'fnl_range': [-1000, 1000],
  'force_alm_gen': False,
  'lensing': False,
  'model_dir': 'data/models',
  'name': 'ulnn

In [33]:
def get_data(start, step, name=None):
    ret = d.skip(start).take(step)

    # # This just fixes the logging output not knowing the dataset size
    ret = ret.apply(tf.data.experimental.assert_cardinality(step))

    ret = ret.cache()
    # ret = ret.batch(batch_size, num_parallel_calls=tf.data.AUTOTUNE, name=name)
    ret = ret.prefetch(tf.data.AUTOTUNE)
    return ret

In [34]:

# Setup the data split as 80/10/10
n = s.total_sims

train_size = int(n * 0.8)
val_size = int(n * 0.1)
test_size = int(n * 0.1)

# loads in the datasets
train_dataset = get_data(0, train_size, "train")
val_dataset = get_data(train_size, val_size, "val")
test_dataset = get_data(train_size + val_size, test_size, "test")

print('Training size:', train_size, 'Validation size:', val_size, 'Test size:', test_size)


Training size: 8000 Validation size: 1000 Test size: 1000


In [35]:
X_train = []
y_train = []
for features, labels in train_dataset:
    # we normalize the data to be between 0 and 1
    min_val = np.min(features)
    max_val = np.max(features)
    features = (features - min_val) / (max_val - min_val)
    X_train.append(features)
    y_train.append(labels)
X_train = np.array(X_train)
y_train = np.array(y_train)

X_test = []
y_test = []
for features, labels in test_dataset:
    min_val = np.min(features)
    max_val = np.max(features)
    features = (features - min_val) / (max_val - min_val)
    X_test.append(features)
    y_test.append(labels)
X_test = np.array(X_test)
y_test = np.array(y_test)

X_val = []
y_val = []
for features, labels in val_dataset:
    min_val = np.min(features)
    max_val = np.max(features)
    features = (features - min_val) / (max_val - min_val)
    X_val.append(features)
    y_val.append(labels)
X_val = np.array(X_val)
y_val = np.array(y_val)

In [36]:
IMG_SHAPE = (s.nside, s.nside, 1)

X_train.shape, y_train.shape, X_test.shape, y_test.shape

((8000, 256, 256, 1), (8000,), (1000, 256, 256, 1), (1000,))

In [37]:
def dice_coefficient(y_true, y_pred, smooth=1.0):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2.0 * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)


def dice_coefficient_loss(y_true, y_pred):
    return -dice_coefficient(y_true, y_pred)

class ReflectionPadding2D(Layer):
    def __init__(self, padding=(1, 1), **kwargs):
        self.padding = tuple(padding)
        self.input_spec = [tf.keras.layers.InputSpec(ndim=4)]
        super(ReflectionPadding2D, self).__init__(**kwargs)

    def compute_output_shape(self, s):
        """If you are using "channels_last" configuration"""
        return (s[0], s[1] + 2 * self.padding[0], s[2] + 2 * self.padding[1], s[3])

    def call(self, x, mask=None):
        w_pad, h_pad = self.padding
        return pad(x, [[0, 0], [h_pad, h_pad], [w_pad, w_pad], [0, 0]], "REFLECT")
    
    def get_config(self):
        config = super().get_config()
        config.update({
            "padding": self.padding,
            "input_spec": self.input_spec,
        })
        return config


def create_localization_module(input_layer, n_filters):
    layer1 = ReflectionPadding2D()(input_layer)
    convolution1 = create_convolution_block(layer1, n_filters)
    return create_convolution_block(convolution1, n_filters, kernel=(1, 1))


def create_up_sampling_module(input_layer, n_filters, size=(2, 2)):
    up_sample = UpSampling2D(size=size, interpolation='bicubic')(input_layer)
    layer1 = ReflectionPadding2D()(up_sample)
    return create_convolution_block(layer1, n_filters)


def create_context_module(
    input_layer, n_level_filters, dropout_rate=0.1, data_format="channels_last"
):
    layer1 = ReflectionPadding2D()(input_layer)
    convolution1 = create_convolution_block(
        input_layer=layer1, n_filters=n_level_filters
    )
    dropout = SpatialDropout2D(rate=dropout_rate, data_format=data_format)(convolution1)
    layer2 = ReflectionPadding2D()(dropout)
    return create_convolution_block(
        input_layer=layer2, n_filters=n_level_filters
    )


def create_convolution_block(
    input_layer,
    n_filters,
    batch_normalization=False,
    kernel=(3, 3),
    activation=LeakyReLU, # maybe try PReLU
    padding="valid",
    strides=(1, 1),
    # instance_normalization=True,
):
    """
    :param strides:
    :param input_layer:
    :param n_filters:
    :param batch_normalization:
    :param kernel:
    :param activation: Keras activation layer to use. (default is 'relu')
    :param padding:
    :return:
    """
    layer = Conv2D(n_filters, kernel, padding=padding, strides=strides)(input_layer)
    if batch_normalization:
        layer = BatchNormalization()(layer)
    # elif instance_normalization:
    #     layer = InstanceNormalization()(layer)
    return Activation("relu")(layer) if activation is None else activation()(layer)

def gated_attention(input_layer, gating_layer, inter_shape):
    theta_x = Conv2D(inter_shape, (2, 2), strides=(1, 1), padding='same')(input_layer)
    g = Conv2D(inter_shape, (2, 2), strides=(2, 2), padding='same')(gating_layer)
    g = UpSampling2D(size=(2, 2))(g)
    add_xg = Add()([theta_x, g])
    psi = Conv2D(1, (1, 1), padding='same', activation='sigmoid')(add_xg)
    return Multiply()([input_layer, psi])

def attn_model(
    inputs,
    depth=5,
    dropout_rate=0.3,
    n_segmentation_levels=3,
    n_labels=1,
    optimizer=Adam,
    initial_learning_rate=5e-4,
    loss_function=dice_coefficient_loss,
    activation_name="relu",
    name=''
):
    """
    This function builds a model proposed by Isensee et al. for the BRATS 2017 competition:
    https://www.cbica.upenn.edu/sbia/Spyridon.Bakas/MICCAI_BraTS/MICCAI_BraTS_2017_proceedings_shortPapers.pdf
    This network is highly similar to the model proposed by Kayalibay et al. "CNN-based Segmentation of Medical
    Imaging Data", 2017: https://arxiv.org/pdf/1701.03056.pdf
    :param inputs:
    :param n_base_filters:
    :param depth:
    :param dropout_rate:
    :param n_segmentation_levels:
    :param n_labels:
    :param optimizer:
    :param initial_learning_rate:
    :param loss_function:
    :param activation_name:
    :return:
    """

    current_layer = inputs
    level_output_layers = []
    level_filters = []
    # n_level_filters = (2**level_number) * n_base_filters
    n_level_filters = 8
    for _ in range(depth):
        level_filters.append(n_level_filters)

        if current_layer is inputs:
            layer = ReflectionPadding2D()(current_layer)
            in_conv = create_convolution_block(layer, n_level_filters)
        else:
            layer = ReflectionPadding2D()(current_layer)
            in_conv = create_convolution_block(layer, n_level_filters, strides=(2, 2))

        context_output_layer = create_context_module(
            in_conv, n_level_filters, dropout_rate=dropout_rate
        )

        summation_layer = Add()([in_conv, context_output_layer])
        level_output_layers.append(summation_layer)
        current_layer = summation_layer

    segmentation_layers = []
    for level_number in range(depth - 2, -1, -1):
        # attention = tf.keras.layers.Attention()([current_layer, current_layer])
        up_sampling = create_up_sampling_module(
            current_layer, level_filters[level_number]
        )

        attention = tf.keras.layers.Attention()([level_output_layers[level_number], up_sampling])

        concatenation_layer = Concatenate()(
            [attention, up_sampling]
        )
        localization_output = create_localization_module(
            concatenation_layer, level_filters[level_number]
        )
        current_layer = localization_output
        if level_number < n_segmentation_levels:
            segmentation_layers.insert(0, Conv2D(8, (1, 1), activation='relu')(current_layer))

    output_layer = None
    for level_number in reversed(range(n_segmentation_levels - 1)):
        segmentation_layer = segmentation_layers[level_number]
        if output_layer is None:
            output_layer = segmentation_layer
        else:
            output_layer = Add()([output_layer, segmentation_layer])

        if level_number > 0:
            output_layer = UpSampling2D(size=(2, 2))(output_layer)

    flat_layer = Flatten()(output_layer)
    # out_layer = Dense(1024, activation='sigmoid')(flat_layer)
    out_layer = Dense(1, activation=None)(flat_layer)

    model = Model(inputs=inputs, outputs=out_layer, name=name)
    return model

In [38]:
model = attn_model(Input(shape=IMG_SHAPE), name='attn_model')

print(model.summary())

Model: "attn_model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_7 (InputLayer)        [(None, 256, 256, 1)]        0         []                            
                                                                                                  
 reflection_padding2d_131 (  (None, 258, 258, 1)          0         ['input_7[0][0]']             
 ReflectionPadding2D)                                                                             
                                                                                                  
 conv2d_166 (Conv2D)         (None, 256, 256, 8)          80        ['reflection_padding2d_131[0][
                                                                    0]']                          
                                                                                         

In [39]:
model.compile(optimizer='rmsprop', 
              loss='mean_squared_error', 
              metrics=['mean_squared_error'])

from tensorflow.keras.callbacks import EarlyStopping
early_stopping = EarlyStopping(monitor='val_loss', patience=10)
history = model.fit(X_train, y_train, 
                    epochs=100,
                    # batch_size=1, 
                    validation_data=(X_test, y_test),
                    callbacks=[early_stopping],)

Epoch 1/100


2023-11-29 12:55:33.014044: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inattn_model/spatial_dropout2d_30/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer





250/250 [==============================] - ETA: 0s - loss: 343393.8438 - mean_squared_error: 343393.8438

250/250 [==============================] - 94s 342ms/step - loss: 343393.8125 - mean_squared_error: 343393.8125 - val_loss: 339896.4062 - val_mean_squared_error: 339896.4062
Epoch 2/100
250/250 [==============================] - ETA: 0s - loss: 339606.9062 - mean_squared_error: 339606.9062

250/250 [==============================] - 86s 344ms/step - loss: 339606.8750 - mean_squared_error: 339606.8750 - val_loss: 339897.4062 - val_mean_squared_error: 339897.4062
Epoch 3/100
250/250 [==============================] - 82s 329ms/step - loss: 339606.5625 - mean_squared_error: 339606.5625 - val_loss: 339898.3125 - val_mean_squared_error: 339898.3125
Epoch 4/100
250/250 [==============================] - 82s 330ms/step - loss: 339606.2188 - mean_squared_error: 339606.2188 - val_loss: 339899.0938 - val_mean_squared_error: 339899.0938
Epoch 5/100
250/250 [==============================] - 83s 330ms/step - loss: 339605.8750 - mean_squared_error: 339605.8750 - val_loss: 339899.8125 - val_mean_squared_error: 339899.8125
Epoch 6/100
250/250 [==============================] - 83s 334ms/step - loss: 339605.5938 - mean_squared_error: 339605.5938 - val_loss: 339900.5000 - val_mean_squared_error: 339900.5000
Epoch 7/100
250/250 [==============================] - 83s 331ms/step - loss: 3396

KeyboardInterrupt: 

In [ ]:

# Plot training & validation loss values
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Test'], loc='upper left')

plt.tight_layout()
plt.show()
plt.savefig('attn_model.png')


In [ ]:
import seaborn as sns
import pandas as pd

# Predict on the validation dataset
y_pred = model.predict(X_val, verbose=1)

# Create a dataframe with true and predicted labels
df = pd.DataFrame({'True Labels': y_val.flatten(), 'Predicted Labels': y_pred.flatten()})

# Create a scatter plot with seaborn
plt.figure(figsize=(12, 6))
sns.scatterplot(data=df, x='True Labels', y='Predicted Labels')
plt.plot([min(y_val), max(y_val)], [min(y_val), max(y_val)], color='red')  # Line for perfect fit

plt.tight_layout()
plt.show()
plt.save('test-2.png')